In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(
    """
        CREATE TABLE IF NOT EXISTS retail_sales_dev.gold.dim_customer (
        CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
        CustomerID STRING,
        Country STRING,
        StartDate DATE,
        EndDate DATE,
        IsCurrent BOOLEAN,
        CreatedTimestamp TIMESTAMP,
        UpdatedTimestamp TIMESTAMP
    ) USING DELTA
    """
)

In [0]:
silver = spark.table("retail_sales_dev.silver.sales_cleaned")

country_counts = silver.groupBy("Country","CustomerID").agg(F.count("*").alias("cnt"))

w = Window.partitionBy("CustomerID").orderBy(F.desc("cnt"))

# display(country_counts)


In [0]:
customer_source = (
    country_counts
    .withColumn("rn",F.row_number().over(w))
    .filter("rn = 1")
    .select("CustomerID","Country")
    .withColumn("UpdatedTimestamp",F.current_timestamp())
)

customer_source.createOrReplaceTempView("customer_source")
# display(customer_source)

In [0]:

# -- Statement 1: close out changed customers' old rows, insert brand-new customers
spark.sql(
    """
    MERGE INTO retail_sales_dev.gold.dim_customer AS T USING
    customer_source AS S ON 
    T.CustomerID = S.CustomerID AND T.IsCurrent = True
    WHEN MATCHED AND T.Country <> S.Country THEN 
    UPDATE SET 
    T.EndDate =  current_date(),
    T.IsCurrent = false,
    T.UpdatedTimestamp = S.UpdatedTimestamp
    WHEN NOT MATCHED THEN
    INSERT(CustomerID, Country, CreatedTimestamp, UpdatedTimestamp,StartDate,IsCurrent)
    VALUES(S.CustomerID, S.Country, current_timestamp(),S.UpdatedTimestamp,current_date(),True)
    """
)
# Statement 2: insert the new "current" row for anyone whose country just changed
spark.sql(
    """
    INSERT INTO retail_sales_dev.gold.dim_customer
    (CustomerID, Country, CreatedTimestamp, UpdatedTimestamp, StartDate, IsCurrent)
    SELECT S.CustomerID, S.Country, current_timestamp(), S.UpdatedTimestamp, current_date(), True
    FROM customer_source AS S
    LEFT ANTI JOIN (
    SELECT CustomerID FROM retail_sales_dev.gold.dim_customer WHERE IsCurrent = True
    ) AS T
    ON S.CustomerID = T.CustomerID
    """
)


In [0]:
%sql
select count(1) from retail_sales_dev.gold.dim_customer